# LLM Experiments QA — Batch API

Build and submit the six QA conditions for Frisian, English, German, and Dutch.

Input files: `Data - QA_Dataset.csv`, `Data - Stories_Metadata.csv`, and `stories.rdf`.


## 1. Configuration

Set `OPENAI_API_KEY` before running the notebook.


In [ ]:
!pip install rdflib

In [ ]:
import csv
import json
import os
from pathlib import Path

import pandas as pd
from openai import OpenAI
from rdflib import Graph, Namespace, RDF, RDFS, OWL
from dotenv import load_dotenv

import time
from google import genai
from google.colab import userdata
from google.genai import types

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
DATA_DIR = Path("Main") if Path("Main/stories.rdf").exists() else Path(".")
QUESTIONS_PATH = DATA_DIR / "Data - QA_Dataset.csv"
STORIES_METADATA_PATH = DATA_DIR / "Data - Stories_Metadata.csv"
RDF_PATH = DATA_DIR / "stories.rdf"
OUTPUT_DIR = DATA_DIR / "batch_qa_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "gpt-5.6-luna"
REASONING_EFFORT = "low"
LANGUAGES = {"fy": "Frisian", "en": "English", "de": "German", "nl": "Dutch"}

SYSTEM_PROMPT = "{ontology_schema}"

OUTPUT_REQUIREMENT = """
Return exactly one valid JSON object with exactly these keys:
{"subject": "...", "predicate": "...", "object": "..."}

Answer the question using only the supplied context. Use "unknown" when a requested value cannot be determined.

The predicate must be the exact ontology property identifier, never its translated or human-readable label. For example, output "appearsIn", not "komt voor in verhaal".

Preserve canonical identifiers exactly, including capitalization, punctuation, and underscores. For example, output "Item_47370", not "Item 47370".

Natural-language subject and object labels may use the language of the question, but canonical story and entity identifiers must not be translated or reformatted.

Before returning the answer, verify that all three keys are present, the JSON is valid, the predicate is an ontology identifier, and identifiers are unchanged. Output JSON only, with no Markdown or explanation.
""".strip()

# Make OpenAI client initialization conditional on API_KEY being set
if API_KEY:
    client = OpenAI(api_key=API_KEY)
    print("OpenAI client initialized successfully.")
else:
    client = None
    print("OpenAI API key not found. OpenAI client is disabled.")

print("Model:", MODEL)
print("Reasoning effort:", REASONING_EFFORT)
print("Output directory:", OUTPUT_DIR.resolve())

OpenAI API key not found. OpenAI client is disabled.
Model: gpt-5.6-luna
Reasoning effort: low
Output directory: /content/batch_qa_output


### 1.1 Gemini Configuration and Initialization

Configure your Google API key to use Gemini models for your QA experiments.

In [ ]:
try:
    GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")

gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
GEMINI_MODEL = "gemini-3.5-flash-lite"

if gemini_client:
    print("Gemini client initialized successfully with model:", GEMINI_MODEL)
else:
    print("Gemini API key not found. Please set GOOGLE_API_KEY in your Colab Secrets or environment variables.")

Gemini client initialized successfully with model: gemini-3.5-flash-lite


## 2. Load data


In [ ]:
CSV_QUESTION_COLUMNS = {"nl": "question_NL", "en": "question_EN", "de": "question_DE", "fy": "question_FY"}
CSV_ANSWER_COLUMNS = {"nl": "Letterlijk antwoord", "en": "obj_EN", "de": "obj_DE", "fy": "obj_FY"}
VB = Namespace("http://example.org/verhalenbank/")

def load_questions(path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))

graph = Graph()
graph.parse(RDF_PATH, format="xml")
question_rows = load_questions(QUESTIONS_PATH)
story_metadata = {row["story_id"]: row for row in load_questions(STORIES_METADATA_PATH)}
questions = []
for row in question_rows:
    for language, question_column in CSV_QUESTION_COLUMNS.items():
        question_text = row.get(question_column, "").strip()
        if question_text:
            questions.append({
                "question_id": row["qa_id"],
                "story_id": row["story_id"],
                "language": language,
                "question": question_text,
                "expected_output": row.get(CSV_ANSWER_COLUMNS[language], "").strip(),
            })

def local_name(value):
    return str(value).rsplit("/", 1)[-1].replace("_", " ")

def rdf_fact_text(story_id):
    story_uri = VB[str(story_id)]
    linked_subjects = {story_uri}
    linked_subjects.update(subject for subject, _, obj in graph.triples((None, VB.appearsIn, story_uri)))
    lines = []
    for subject in sorted(linked_subjects, key=str):
        for predicate, obj in sorted(graph.predicate_objects(subject), key=lambda pair: (str(pair[0]), str(pair[1]))):
            if predicate in {RDF.type, RDFS.label} and subject != story_uri:
                continue
            lines.append(f"{local_name(subject)} --{local_name(predicate)}--> {local_name(obj)}")
    if not lines:
        raise ValueError(f"No RDF facts found for story {story_id!r}.")
    return "\n".join(lines)

schema_subjects = (set(graph.subjects(RDF.type, OWL.Class)) |
                   set(graph.subjects(RDF.type, OWL.ObjectProperty)) |
                   set(graph.subjects(RDF.type, OWL.DatatypeProperty)))
schema_graph = Graph()
for subject in schema_subjects:
    for triple in graph.triples((subject, None, None)):
        schema_graph.add(triple)
ontology_text = schema_graph.serialize(format="turtle")
full_ontology_text = graph.serialize(format="turtle")
story_by_id = {}
for row in question_rows:
    story_id = str(row["story_id"])
    if story_id not in story_metadata:
        raise ValueError(f"Story {story_id!r} is missing from the metadata CSV.")
    story_by_id[story_id] = {
        "text": story_metadata[story_id].get("main_text", "").strip(),
        "rdf_facts": rdf_fact_text(story_id),
    }
    if not story_by_id[story_id]["text"]:
        raise ValueError(f"Story {story_id!r} has no main_text in the metadata CSV.")

print(f"Loaded {len(questions)} translated questions from {len(question_rows)} CSV rows.")
print(f"Loaded {len(graph):,} RDF triples and {len(story_by_id)} referenced stories.")


Loaded 240 translated questions from 60 CSV rows.
Loaded 534 RDF triples and 30 referenced stories.


## 3. Build the six conditions


In [ ]:
CONDITIONS = [
    {"id": "zero_shot", "label": "Zero-shot", "include_ontology": False, "include_target_story": False, "few_shot": False},
    {"id": "zero_shot_ontology", "label": "Zero-shot - Ontology", "include_ontology": True, "include_target_story": False, "few_shot": False},
    {"id": "one_shot", "label": "One-shot", "include_ontology": False, "include_target_story": True, "few_shot": False},
    {"id": "one_shot_ontology", "label": "One-shot - Ontology", "include_ontology": True, "include_target_story": True, "few_shot": False},
    {"id": "few_shot", "label": "Few-shot", "include_ontology": False, "include_target_story": True, "few_shot": True},
    {"id": "few_shot_ontology", "label": "Few-shot - Ontology", "include_ontology": True, "include_target_story": True, "few_shot": True},
]

def select_example_stories(question, target_story_id):
    target_id = str(target_story_id)
    candidates = [q for q in questions if q["language"] == question["language"] and str(q["story_id"]) != target_id]
    if len(candidates) < 2:
        raise ValueError(f"At least two non-target CSV questions are required for language {question['language']!r}.")
    return candidates[0], candidates[1]

def get_target_story(question):
    story_id = str(question["story_id"])
    if story_id not in story_by_id:
        raise ValueError(f"Story {story_id!r} referenced by question {question['question_id']} was not found in the RDF graph.")
    return story_by_id[story_id]

def make_system_prompt():
    return SYSTEM_PROMPT.format(ontology_schema=ontology_text)

def make_user_prompt(question, condition):
    sections = [OUTPUT_REQUIREMENT]
    if condition["include_ontology"]:
        sections.append("Complete filled ontology graph. Use it as the source of truth for entity identifiers and relations:\n" + full_ontology_text)
    if condition["include_target_story"]:
        target = get_target_story(question)
        sections.append("Relevant target story (the story referred to by the question):\nStory ID: " + str(question["story_id"]) + "\nStory main text:\n" + target["text"] + "\n\nRDF facts:\n" + target["rdf_facts"])
    if condition["few_shot"]:
        _, wrong = select_example_stories(question, question["story_id"])
        wrong_story = story_by_id[str(wrong["story_id"])]
        sections.append("Distractor story (unrelated to the question; ignore it unless the question explicitly refers to it):\nStory ID: " + str(wrong["story_id"]) + "\nStory main text:\n" + wrong_story["text"] + "\n\nRDF facts:\n" + wrong_story["rdf_facts"])
    sections.append("Question:\n" + question["question"])
    return "\n\n".join(sections)

def build_requests():
    requests = []
    for question in questions:
        for condition in CONDITIONS:
            if condition["include_target_story"] or condition["few_shot"]:
                get_target_story(question)
            if condition["few_shot"]:
                select_example_stories(question, question["story_id"])
            custom_id = "__".join([str(question["question_id"]), question["language"], condition["id"]])
            requests.append({
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": MODEL,
                    "input": [
                        {"role": "system", "content": make_system_prompt()},
                        {"role": "user", "content": make_user_prompt(question, condition)},
                    ],
                    "reasoning": {"effort": REASONING_EFFORT},
                    "tool_choice": "none",
                },
            })
    return requests

requests = build_requests()
print(f"Prepared {len(requests)} requests from the CSV questions.")
display(pd.DataFrame([{"custom_id": r["custom_id"], "model": r["body"]["model"], "reasoning": r["body"]["reasoning"]["effort"]} for r in requests[:10]]))


Prepared 1440 requests from the CSV questions.


,custom_id,model,reasoning
0,Marten_01_1__nl__zero_shot,gpt-5.6-luna,low
1,Marten_01_1__nl__zero_shot_ontology,gpt-5.6-luna,low
2,Marten_01_1__nl__one_shot,gpt-5.6-luna,low
3,Marten_01_1__nl__one_shot_ontology,gpt-5.6-luna,low
4,Marten_01_1__nl__few_shot,gpt-5.6-luna,low
5,Marten_01_1__nl__few_shot_ontology,gpt-5.6-luna,low
6,Marten_01_1__en__zero_shot,gpt-5.6-luna,low
7,Marten_01_1__en__zero_shot_ontology,gpt-5.6-luna,low
8,Marten_01_1__en__one_shot,gpt-5.6-luna,low
9,Marten_01_1__en__one_shot_ontology,gpt-5.6-luna,low


## 4. Export requests


In [ ]:
jsonl_path = OUTPUT_DIR / "qa_batch_requests.jsonl"

if not jsonl_path.exists():
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.touch()

with jsonl_path.open("w", encoding="utf-8") as file:
    for request in requests:
        file.write(json.dumps(request, ensure_ascii=False) + "\n")

print(json.dumps(requests[0], ensure_ascii=False, indent=2))


{
  "custom_id": "Marten_01_1__nl__zero_shot",
  "method": "POST",
  "url": "/v1/responses",
  "body": {
    "model": "gpt-5.6-luna",
    "input": [
      {
        "role": "system",
        "content": "@prefix owl: <http://www.w3.org/2002/07/owl#> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n\n<http://example.org/verhalenbank/Action> a owl:Class ;\n    rdfs:label \"Handeling\"@nl .\n\n<http://example.org/verhalenbank/appearsIn> a owl:ObjectProperty ;\n    rdfs:label \"karakter komt voor in verhaal\"@nl ;\n    rdfs:domain <http://example.org/verhalenbank/Character> ;\n    rdfs:range <http://example.org/verhalenbank/Story> .\n\n<http://example.org/verhalenbank/defeats> a owl:ObjectProperty ;\n    rdfs:label \"karakter verslaat of misleidt ander karakter\"@nl ;\n    rdfs:domain <http://example.org/verhalenbank/Character> ;\n    rdfs:range <http://example.org/verhalenbank/Character> .\n\n<http://example.org/verhalenbank/di

## 5. Submit a test batch

Turn CONFIRM_TEST_SUBMISSION to true if you are ready to submit


In [ ]:
# Test run: one question in four languages x six conditions = 24 requests.
TEST_QA_ID = question_rows[20]["qa_id"]
CONFIRM_TEST_SUBMISSION = False

test_questions = [q for q in questions if q["question_id"] == TEST_QA_ID]
if len(test_questions) != len(LANGUAGES) or {q["language"] for q in test_questions} != set(LANGUAGES):
    raise ValueError(f"QA row {TEST_QA_ID!r} must contain exactly one question in each expected language.")

test_requests = []
for question in test_questions:
    for condition in CONDITIONS:
        if condition["include_target_story"] or condition["few_shot"]:
            get_target_story(question)
        if condition["few_shot"]:
            select_example_stories(question, question["story_id"])
        test_requests.append({
            "custom_id": "__".join(["test", str(question["question_id"]), question["language"], condition["id"]]),
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": MODEL,
                "input": [
                    {"role": "system", "content": make_system_prompt()},
                    {"role": "user", "content": make_user_prompt(question, condition)},
                ],
                "reasoning": {"effort": REASONING_EFFORT},
                "tool_choice": "none",
            },
        })

test_jsonl_path = OUTPUT_DIR / "test_qa_batch_requests.jsonl"
with test_jsonl_path.open("w", encoding="utf-8") as file:
    for request in test_requests:
        file.write(json.dumps(request, ensure_ascii=False) + "\n")

print(f"Prepared {len(test_requests)} requests for QA row {TEST_QA_ID!r}.")
print("Wrote:", test_jsonl_path)

if CONFIRM_TEST_SUBMISSION:
    with test_jsonl_path.open("rb") as file:
        uploaded_test_file = client.files.create(file=file, purpose="batch")
    test_batch = client.batches.create(
        input_file_id=uploaded_test_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={"experiment": "LLM experiments QA test run", "qa_id": str(TEST_QA_ID)},
    )
    test_batch_info = {
        "batch_id": test_batch.id,
        "input_file_id": uploaded_test_file.id,
        "status": test_batch.status,
        "qa_id": TEST_QA_ID,
        "request_count": len(test_requests),
    }
    (OUTPUT_DIR / "test_batch_submission.json").write_text(json.dumps(test_batch_info, indent=2), encoding="utf-8")
    print(json.dumps(test_batch_info, indent=2))
else:
    print("Not submitted. Set CONFIRM_TEST_SUBMISSION = True and rerun this cell to submit one batch.")


Prepared 24 requests for QA row 'Jasper_01_1'.
Wrote: batch_qa_output/test_qa_batch_requests.jsonl
Not submitted. Set CONFIRM_TEST_SUBMISSION = True and rerun this cell to submit one batch.


In [ ]:
TEST_QA_ID = question_rows[20]["qa_id"]
CONFIRM_GEMINI_TEST = False

test_questions = [q for q in questions if q["question_id"] == TEST_QA_ID]
if len(test_questions) != len(LANGUAGES) or {q["language"] for q in test_questions} != set(LANGUAGES):
    raise ValueError(f"QA row {TEST_QA_ID!r} must contain exactly one question in each expected language.")

gemini_test_requests = []
for question in test_questions:
    for condition in CONDITIONS:
        if condition["include_target_story"] or condition["few_shot"]:
            get_target_story(question)
        if condition["few_shot"]:
            select_example_stories(question, question["story_id"])
        custom_id = "__".join(["test", str(question["question_id"]), question["language"], condition["id"]])
        gemini_test_requests.append({
            "key": custom_id,
            "request": {
                "contents": [{"role": "user", "parts": [{"text": make_user_prompt(question, condition)}]}],
                "systemInstruction": {"parts": [{"text": make_system_prompt()}]},
                "generationConfig": {
                    "responseMimeType": "application/json"
                }
            }
        })

gemini_test_jsonl_path = OUTPUT_DIR / "test_gemini_batch_requests.jsonl"
with gemini_test_jsonl_path.open("w", encoding="utf-8") as file:
    for req in gemini_test_requests:
        file.write(json.dumps(req, ensure_ascii=False) + "\n")

print(f"Prepared {len(gemini_test_requests)} requests for Gemini test run.")
print("Wrote:", gemini_test_jsonl_path)

if CONFIRM_GEMINI_TEST:
    if not gemini_client:
        raise ValueError("Gemini client is not initialized. Check GOOGLE_API_KEY.")

    uploaded_batch_file = gemini_client.files.upload(
        file=gemini_test_jsonl_path,
        config=types.UploadFileConfig(display_name="test-gemini-batch", mime_type="jsonl")
    )
    print(f"Uploaded input file: {uploaded_batch_file.name}")

    batch_job = gemini_client.batches.create(
        model=GEMINI_MODEL,
        src=uploaded_batch_file.name,
        config={"display_name": f"test-gemini-qa-{TEST_QA_ID}"}
    )
    print(f"Created batch job: {batch_job.name}")

    completed_states = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}
    while True:
        job = gemini_client.batches.get(name=batch_job.name)
        state_name = job.state.name if hasattr(job.state, "name") else str(job.state)
        print(f"Job status: {state_name}")
        if state_name in completed_states:
            break
        time.sleep(15)

    if state_name == "JOB_STATE_SUCCEEDED":
        result_file_name = job.dest.file_name
        print(f"Downloading results from: {result_file_name}")
        result_content = gemini_client.files.download(file=result_file_name)

        gemini_result_path = OUTPUT_DIR / "test_gemini_batch_results.jsonl"
        gemini_result_path.write_bytes(result_content)
        print(f"Saved results to: {gemini_result_path}")
    else:
        print(f"Batch failed with status: {state_name}")
else:
    print("Not submitted. Set CONFIRM_GEMINI_TEST = True and rerun this cell.")

Prepared 24 requests for Gemini test run.
Wrote: batch_qa_output/test_gemini_batch_requests.jsonl
Not submitted. Set CONFIRM_GEMINI_TEST = True and rerun this cell.


## 6. Submit the real batch

Turn CONFIRM_SUBMISSION to true if you are ready to submit

In [ ]:
CONFIRM_SUBMISSION = False

if not CONFIRM_SUBMISSION:
    print("Not submitted. Set CONFIRM_SUBMISSION = True after reviewing the request JSONL.")
else:
    with jsonl_path.open("rb") as file:
        uploaded_file = client.files.create(file=file, purpose="batch")

    batch = client.batches.create(
        input_file_id=uploaded_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "experiment": "LLM experiments QA",
            "model": MODEL,
            "reasoning_effort": REASONING_EFFORT,
        },
    )

    batch_info = {
        "batch_id": batch.id,
        "input_file_id": uploaded_file.id,
        "status": batch.status,
        "request_count": len(requests),
    }
    (OUTPUT_DIR / "batch_submission.json").write_text(
        json.dumps(batch_info, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(batch_info, indent=2))


Not submitted. Set CONFIRM_SUBMISSION = True after reviewing the request JSONL.


In [ ]:
import time
from google.genai import types

CONFIRM_GEMINI_FULL_SUBMISSION = False

gemini_all_requests = []
for question in questions:
    for condition in CONDITIONS:
        if condition["include_target_story"] or condition["few_shot"]:
            get_target_story(question)
        if condition["few_shot"]:
            select_example_stories(question, question["story_id"])
        custom_id = "__".join([str(question["question_id"]), question["language"], condition["id"]])
        gemini_all_requests.append({
            "key": custom_id,
            "request": {
                "contents": [{"role": "user", "parts": [{"text": make_user_prompt(question, condition)}]}],
                "systemInstruction": {"parts": [{"text": make_system_prompt()}]},
                "generationConfig": {
                    "responseMimeType": "application/json"
                }
            }
        })

gemini_full_jsonl_path = OUTPUT_DIR / "gemini_batch_requests.jsonl"
with gemini_full_jsonl_path.open("w", encoding="utf-8") as file:
    for req in gemini_all_requests:
        file.write(json.dumps(req, ensure_ascii=False) + "\n")

print(f"Prepared {len(gemini_all_requests)} requests for Gemini full dataset.")
print("Wrote:", gemini_full_jsonl_path)

if CONFIRM_GEMINI_FULL_SUBMISSION:
    if not gemini_client:
        raise ValueError("Gemini client is not initialized. Check GOOGLE_API_KEY.")

    uploaded_full_file = gemini_client.files.upload(
        file=gemini_full_jsonl_path,
        config=types.UploadFileConfig(display_name="gemini-full-batch", mime_type="jsonl")
    )
    print(f"Uploaded input file: {uploaded_full_file.name}")

    full_batch_job = gemini_client.batches.create(
        model=GEMINI_MODEL,
        src=uploaded_full_file.name,
        config={"display_name": "gemini-qa-full-dataset"}
    )
    print(f"Created batch job: {full_batch_job.name}")

    gemini_submission_info = {
        "job_name": full_batch_job.name,
        "input_file": uploaded_full_file.name,
        "request_count": len(gemini_all_requests)
    }
    (OUTPUT_DIR / "gemini_batch_submission.json").write_text(
        json.dumps(gemini_submission_info, indent=2), encoding="utf-8"
    )

    completed_states = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}
    while True:
        job = gemini_client.batches.get(name=full_batch_job.name)
        state_name = job.state.name if hasattr(job.state, "name") else str(job.state)
        print(f"Job status: {state_name}")
        if state_name in completed_states:
            break
        time.sleep(30)

    if state_name == "JOB_STATE_SUCCEEDED":
        result_file_name = job.dest.file_name
        print(f"Downloading results from: {result_file_name}")
        result_content = gemini_client.files.download(file=result_file_name)
        gemini_result_path = OUTPUT_DIR / "gemini_batch_results.jsonl"
        gemini_result_path.write_bytes(result_content)
        print(f"Saved results to: {gemini_result_path}")
    else:
        print(f"Batch failed with status: {state_name}")
else:
    print("Not submitted. Set CONFIRM_GEMINI_FULL_SUBMISSION = True to execute.")

Prepared 1440 requests for Gemini full dataset.
Wrote: batch_qa_output/gemini_batch_requests.jsonl
Uploaded input file: files/tbxp65iupzn7
Created batch job: batches/7ixmdpl7zjz4luqgj6upfovid5tn6pvpszj1
Job status: JOB_STATE_RUNNING
Job status: JOB_STATE_RUNNING
Job status: JOB_STATE_RUNNING
Job status: JOB_STATE_RUNNING
Job status: JOB_STATE_RUNNING
Job status: JOB_STATE_RUNNING
Job status: JOB_STATE_RUNNING
Job status: JOB_STATE_SUCCEEDED
Saved results to: batch_qa_output/gemini_batch_results.jsonl


## 7. Retrieve results

switch TEST_RUN too true if you want to retrieve the results of the test batch instead


In [ ]:
TEST_RUN = False
submission_name = "test_batch_submission.json" if TEST_RUN else "batch_submission.json"
result_name = "test_batch_results.jsonl" if TEST_RUN else "batch_results.jsonl"
error_name = "test_batch_errors.jsonl" if TEST_RUN else "batch_errors.jsonl"
submission_path = OUTPUT_DIR / submission_name

if not submission_path.exists():
    print(f"No saved submission found at {submission_path}. Submit the batch first.")
else:
    saved = json.loads(submission_path.read_text(encoding="utf-8"))
    batch = client.batches.retrieve(saved["batch_id"])
    print("Batch:", batch.id)
    print("Status:", batch.status)
    print("Counts:", batch.request_counts)

    if batch.output_file_id:
        output_content = client.files.content(batch.output_file_id).content
        output_path = OUTPUT_DIR / result_name
        output_path.write_bytes(output_content)
        print("Saved results to", output_path)

    if batch.error_file_id:
        error_content = client.files.content(batch.error_file_id).content
        error_path = OUTPUT_DIR / error_name
        error_path.write_bytes(error_content)
        print("Saved errors to", error_path)

No saved submission found at batch_qa_output/batch_submission.json. Submit the batch first.


## 8. Analyze all batch results

In [ ]:
import html
import json
from pathlib import Path
import pandas as pd

PROVIDER = "gemini"  # "gemini" of "openai"
TEST_RUN = False      # True for test, False for complete run

if PROVIDER == "gemini":
    RESULT_FILE_NAME = "test_gemini_batch_results.jsonl" if TEST_RUN else "gemini_batch_results.jsonl"
    ERROR_FILE_NAME = "test_gemini_batch_errors.jsonl" if TEST_RUN else "gemini_batch_errors.jsonl"
    REQUEST_FILE_NAMES = ["test_gemini_batch_requests.jsonl" if TEST_RUN else "gemini_batch_requests.jsonl"]
else:
    RESULT_FILE_NAME = "test_batch_results.jsonl" if TEST_RUN else "batch_results.jsonl"
    ERROR_FILE_NAME = "test_batch_errors.jsonl" if TEST_RUN else "batch_errors.jsonl"
    REQUEST_FILE_NAMES = ["test_qa_batch_requests.jsonl"] if TEST_RUN else ["qa_batch_requests.jsonl", "test_qa_batch_requests.jsonl"]


def load_jsonl(path):
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def normalize_answer(value):
    normalized = html.unescape(str(value or "")).strip().casefold().replace("_", " ")
    return " ".join(normalized.split())


def split_answer_alternatives(value):
    return [part.strip() for part in html.unescape(str(value or "")).split("/") if part.strip()]


def build_predicate_aliases():
    aliases = {}
    for predicate in set(graph.predicates()):
        canonical = normalize_answer(local_name(predicate))
        aliases[canonical] = canonical
        for label in graph.objects(predicate, RDFS.label):
            label_normalized = normalize_answer(label)
            aliases[label_normalized] = canonical
            for prefix in ("karakter ", "verhaal ", "iets "):
                if label_normalized.startswith(prefix):
                    aliases[label_normalized[len(prefix):]] = canonical
    return aliases


predicate_aliases = build_predicate_aliases()


def answers_match(predicted, expected, field):
    if field == "predicate":
        predicted_normalized = normalize_answer(predicted)
        expected_normalized = normalize_answer(expected)
        predicted_normalized = predicate_aliases.get(predicted_normalized, predicted_normalized)
        expected_normalized = predicate_aliases.get(expected_normalized, expected_normalized)
    else:
        predicted_normalized = normalize_answer(predicted)
        expected_alternatives = [normalize_answer(part) for part in split_answer_alternatives(expected)]
        return any(alternative and alternative in predicted_normalized for alternative in expected_alternatives)
    return bool(expected_normalized) and expected_normalized in predicted_normalized


def parse_custom_id(custom_id):
    parts = str(custom_id).split("__")
    language_positions = [index for index, part in enumerate(parts) if part in LANGUAGES]
    if not language_positions:
        return None, None, None
    language_index = language_positions[-1]
    return "__".join(parts[:language_index]), parts[language_index], "__".join(parts[language_index + 1:])


def response_text(record):
    response_obj = record.get("response") or {}
    candidates = response_obj.get("candidates") or []
    if candidates:
        parts = (candidates[0].get("content") or {}).get("parts") or []
        for part in parts:
            if "text" in part:
                return part["text"]

    body = (response_obj.get("body") or {})
    for output_item in body.get("output") or []:
        for content_item in output_item.get("content") or []:
            if content_item.get("type") == "output_text":
                return content_item.get("text", "")
    return ""


def parse_model_answer(text):
    try:
        value = json.loads(text)
    except (TypeError, json.JSONDecodeError):
        return {}
    return value if isinstance(value, dict) else {}


result_path = OUTPUT_DIR / RESULT_FILE_NAME
error_path = OUTPUT_DIR / ERROR_FILE_NAME
result_records = load_jsonl(result_path)
error_records = load_jsonl(error_path)
all_records = result_records + error_records

request_records = []
for request_name in REQUEST_FILE_NAMES:
    candidate_records = load_jsonl(OUTPUT_DIR / request_name)
    if candidate_records:
        request_records = candidate_records
        break
request_by_id = {(record.get("custom_id") or record.get("key")): record for record in request_records}

CSV_SUBJECT_COLUMNS = {"nl": "subj_NL", "en": "subj_EN", "de": "subj_DE", "fy": "subj_FY"}
CSV_OBJECT_COLUMNS = {"nl": "obj_NL", "en": "obj_EN", "de": "obj_DE", "fy": "obj_FY"}

reference_by_key = {}
for row in question_rows:
    for language in LANGUAGES:
        question = row.get(CSV_QUESTION_COLUMNS[language], "").strip()
        if question:
            reference_by_key[(row["qa_id"], language)] = {
                "question": question,
                "story_id": row["story_id"],
                "subject": row.get(CSV_SUBJECT_COLUMNS[language], "").strip(),
                "predicate": row.get("predicate", "").strip(),
                "object": row.get(CSV_OBJECT_COLUMNS[language], "").strip(),
                "expected_output": row.get(CSV_ANSWER_COLUMNS[language], "").strip(),
            }


def reference_for_record(record, references, requests):
    custom_id = record.get("custom_id") or record.get("key", "")
    question_id, language, strategy = parse_custom_id(custom_id)
    if question_id and question_id.startswith("test__"):
        question_id = question_id[len("test__"):]

    candidates = []
    if question_id and language:
        candidates.append(references.get((question_id, language)))
        candidates.append(references.get((question_id.rsplit("__", 1)[-1], language)))

    request = requests.get(custom_id, {})
    user_content = ""
    for message in ((request.get("body") or {}).get("input") or []):
        if message.get("role") == "user":
            user_content = message.get("content", "")
            break
    if not user_content:
        for content in ((request.get("request") or {}).get("contents") or []):
            if content.get("role") == "user":
                parts = content.get("parts") or []
                user_content = " ".join(p.get("text", "") for p in parts if "text" in p)
                break

    for (reference_id, reference_language), reference in references.items():
        if reference_language == language and reference["question"] in user_content:
            candidates.append(reference)
            question_id = reference_id
            break

    reference = next((candidate for candidate in candidates if candidate), {})
    return question_id, language, strategy, reference


def matching_reference_languages(predicted, question_id, field):
    matches = []
    if field == "predicate":
        return matches
    for candidate_language in LANGUAGES:
        candidate = reference_by_key.get((question_id, candidate_language), {})
        expected_value = candidate.get(field, "")
        if expected_value and answers_match(predicted, expected_value, field):
            matches.append(candidate_language)
    return matches


analysis_rows = []
for record in all_records:
    custom_id = record.get("custom_id") or record.get("key", "")
    question_id, language, strategy, reference = reference_for_record(record, reference_by_key, request_by_id)
    response = record.get("response") or {}
    body = response.get("body") or {}
    predicted_text = response_text(record)
    predicted = parse_model_answer(predicted_text)
    expected = {key: reference.get(key, "") for key in ("subject", "predicate", "object")}

    requested_language_correct = {
        key: answers_match(predicted.get(key), expected[key], key) for key in expected
    }
    matched_languages = {
        key: matching_reference_languages(predicted.get(key), question_id, key)
        for key in ("subject", "object")
    }
    wrong_language_accepted = {
        key: bool(matched_languages[key]) and language not in matched_languages[key]
        for key in ("subject", "object")
    }
    slot_correct = {
        key: (bool(matched_languages[key]) if key in matched_languages else requested_language_correct[key])
        for key in expected
    }
    requested_language_triple_correct = all(requested_language_correct.values())
    triple_correct = all(slot_correct.values())
    accepted_via_wrong_language = triple_correct and not requested_language_triple_correct

    created_at = body.get("created_at")
    completed_at = body.get("completed_at")
    duration_seconds = completed_at - created_at if isinstance(created_at, (int, float)) and isinstance(completed_at, (int, float)) else None

    usage = body.get("usage") or {}
    usage_meta = response.get("usageMetadata") or {}
    input_tokens = usage.get("input_tokens") or usage_meta.get("promptTokenCount")
    output_tokens = usage.get("output_tokens") or usage_meta.get("candidatesTokenCount")
    total_tokens = usage.get("total_tokens") or usage_meta.get("totalTokenCount")

    analysis_rows.append({
        "custom_id": custom_id, "question_id": question_id, "story_id": reference.get("story_id", ""),
        "language": language or "unknown", "strategy": strategy or "unknown", "question": reference.get("question", ""),
        "correct": triple_correct, "requested_language_correct": requested_language_triple_correct,
        "accepted_via_wrong_language": accepted_via_wrong_language,
        "subject_correct": slot_correct["subject"], "predicate_correct": slot_correct["predicate"],
        "object_correct": slot_correct["object"],
        "subject_requested_language_correct": requested_language_correct["subject"],
        "object_requested_language_correct": requested_language_correct["object"],
        "subject_wrong_language_accepted": wrong_language_accepted["subject"],
        "object_wrong_language_accepted": wrong_language_accepted["object"],
        "subject_matched_languages": matched_languages["subject"],
        "object_matched_languages": matched_languages["object"],
        "predicted": predicted, "expected": expected, "raw_response": predicted_text,
        "status_code": response.get("status_code"), "response_status": body.get("status"),
        "duration_seconds": duration_seconds, "input_tokens": input_tokens,
        "output_tokens": output_tokens, "total_tokens": total_tokens,
        "reference_found": bool(reference), "request_found": custom_id in request_by_id,
    })

results_df = pd.DataFrame(analysis_rows)
if results_df.empty:
    raise FileNotFoundError(f"Geen resultaten gevonden in {result_path}.")

results_df["incorrect"] = ~results_df["correct"]
results_df["requested_language_incorrect"] = ~results_df["requested_language_correct"]
print(f"Analyzed {len(results_df):,} result records from {result_path.name}.")

overall = pd.DataFrame([{
    "requests": len(results_df),
    "correct": int(results_df["correct"].sum()),
    "incorrect": int(results_df["incorrect"].sum()),
    "accuracy": results_df["correct"].mean(),
    "error_rate": results_df["incorrect"].mean(),
    "references_found": int(results_df["reference_found"].sum()),
    "requested_language_correct": int(results_df["requested_language_correct"].sum()),
    "requested_language_accuracy": results_df["requested_language_correct"].mean(),
    "accepted_via_wrong_language": int(results_df["accepted_via_wrong_language"].sum()),
}])
print("\nOverall results (accept any language's subject/object label):")
display(overall.round(3))

field_accuracy = pd.DataFrame([{
    "subject_correct_any_language": int(results_df["subject_correct"].sum()),
    "subject_accuracy_any_language": results_df["subject_correct"].mean(),
    "subject_requested_language_correct": int(results_df["subject_requested_language_correct"].sum()),
    "subject_accepted_only_via_other_language": int(results_df["subject_wrong_language_accepted"].sum()),
    "predicate_correct": int(results_df["predicate_correct"].sum()),
    "predicate_accuracy": results_df["predicate_correct"].mean(),
    "object_correct_any_language": int(results_df["object_correct"].sum()),
    "object_accuracy_any_language": results_df["object_correct"].mean(),
    "object_requested_language_correct": int(results_df["object_requested_language_correct"].sum()),
    "object_accepted_only_via_other_language": int(results_df["object_wrong_language_accepted"].sum()),
}])
print("\nField-level results (with requested-language comparisons):")
display(field_accuracy.round(3))

print("\nAnswers accepted only through another language, by question language:")
wrong_language_by_language = results_df.groupby("language", dropna=False).agg(
    requests=("custom_id", "size"),
    subject_accepted_only_via_other_language=("subject_wrong_language_accepted", "sum"),
    object_accepted_only_via_other_language=("object_wrong_language_accepted", "sum"),
    triples_rescued_by_other_language=("accepted_via_wrong_language", "sum"),
).reset_index()
display(wrong_language_by_language)

numeric_columns = [col for col in ["duration_seconds", "input_tokens", "output_tokens", "total_tokens"] if results_df[col].notna().any()]
if numeric_columns:
    print("\nRequest measurements:")
    display(results_df[numeric_columns].agg(["count", "mean", "median", "min", "max"]).round(3))

agg_dict = {
    "requests": ("custom_id", "size"),
    "correct_any_language": ("correct", "sum"),
    "accuracy_any_language": ("correct", "mean"),
    "correct_requested_language": ("requested_language_correct", "sum"),
    "triples_rescued_by_other_language": ("accepted_via_wrong_language", "sum"),
    "subject_accepted_only_via_other_language": ("subject_wrong_language_accepted", "sum"),
    "object_accepted_only_via_other_language": ("object_wrong_language_accepted", "sum"),
    "avg_input_tokens": ("input_tokens", "mean"),
    "avg_output_tokens": ("output_tokens", "mean"),
    "avg_total_tokens": ("total_tokens", "mean"),
}
if "duration_seconds" in numeric_columns:
    agg_dict["avg_duration_seconds"] = ("duration_seconds", "mean")
    agg_dict["median_duration_seconds"] = ("duration_seconds", "median")

for group_name in ("language", "strategy"):
    grouped = results_df.groupby(group_name, dropna=False).agg(**agg_dict).reset_index()
    print(f"\nStatistics by {group_name}:")
    display(grouped.round(3))

question_agg = {
    "requests": ("custom_id", "size"),
    "correct": ("correct", "sum"),
    "incorrect": ("incorrect", "sum"),
    "accuracy": ("correct", "mean"),
    "avg_total_tokens": ("total_tokens", "mean"),
}
if "duration_seconds" in numeric_columns:
    question_agg["avg_duration_seconds"] = ("duration_seconds", "mean")

question_stats = results_df.groupby(["question_id", "language"], dropna=False).agg(**question_agg).reset_index()
print("\nStatistics per question:")
display(question_stats.round(3))

print("\nMissing references:", int((~results_df["reference_found"]).sum()))
print("Missing request records:", int((~results_df["request_found"]).sum()))

Analyzed 1,440 result records from gemini_batch_results.jsonl.

Overall results (accept any language's subject/object label):


,requests,correct,incorrect,accuracy,error_rate,references_found,requested_language_correct,requested_language_accuracy,accepted_via_wrong_language
0,1440,654,786,0.454,0.546,1440,258,0.179,396



Field-level results (with requested-language comparisons):


,subject_correct_any_language,subject_accuracy_any_language,subject_requested_language_correct,subject_accepted_only_via_other_language,predicate_correct,predicate_accuracy,object_correct_any_language,object_accuracy_any_language,object_requested_language_correct,object_accepted_only_via_other_language
0,766,0.532,476,290,970,0.674,772,0.536,367,405



Answers accepted only through another language, by question language:


...


Statistics by strategy:


,strategy,requests,correct_any_language,accuracy_any_language,correct_requested_language,triples_rescued_by_other_language,subject_accepted_only_via_other_language,object_accepted_only_via_other_language,avg_input_tokens,avg_output_tokens,avg_total_tokens
0,few_shot,240,78,0.325,40,38,20,53,3498.738,26.190,3524.054
1,few_shot_ontology,240,190,0.792,70,120,89,113,10342.738,33.377,10375.975
2,one_shot,240,70,0.292,36,34,20,48,2599.071,27.175,2626.246
3,one_shot_ontology,240,193,0.804,70,123,92,116,9443.071,33.400,9476.471
4,zero_shot,240,0,0.000,0,0,4,0,1799.204,24.529,1823.733
5,zero_shot_ontology,240,123,0.512,42,81,65,75,8643.204,30.438,8673.642



Statistics per question:


,question_id,language,requests,correct,incorrect,accuracy,avg_total_tokens
0,Fleur_01_1,de,6,2,4,0.333,5719.167
1,Fleur_01_1,en,6,3,3,0.500,5724.500
2,Fleur_01_1,fy,6,1,5,0.167,5719.667
3,Fleur_01_1,nl,6,3,3,0.500,5724.667
4,Fleur_01_2,de,6,4,2,0.667,5715.000
...,...,...,...,...,...,...,...
235,Marten_10_1,nl,6,3,3,0.500,6084.000
236,Marten_10_2,de,6,3,3,0.500,6080.333
237,Marten_10_2,en,6,4,2,0.667,6078.833
238,Marten_10_2,fy,6,3,3,0.500,6092.833



Missing references: 0
Missing request records: 0


In [ ]:
import json
import pandas as pd

incorrect_df = results_df[results_df["incorrect"]].copy()
print(f"Number of wrong predictions: {len(incorrect_df)} of {len(results_df)}")

display_columns = ["custom_id", "language", "strategy", "expected", "predicted"]
pd.set_option("display.max_colwidth", None)
display(incorrect_df[display_columns])

# Set CUSTOM_ID to an exact custom_id, or set ENTRY_INDEX to a zero-based result index.
CUSTOM_ID = ""
ENTRY_INDEX = 0

if not all_records:
    raise FileNotFoundError(f"No result records found at {result_path}.")
if CUSTOM_ID:
    matching = [record for record in all_records if (record.get("custom_id") or record.get("key")) == CUSTOM_ID]
    if not matching:
        raise KeyError(f"No result entry has custom_id={CUSTOM_ID!r}.")
    inspection_record = matching[0]
else:
    if not 0 <= ENTRY_INDEX < len(all_records):
        raise IndexError(f"ENTRY_INDEX must be between 0 and {len(all_records) - 1}.")
    inspection_record = all_records[ENTRY_INDEX]

inspection_custom_id = inspection_record.get("custom_id") or inspection_record.get("key", "")
inspection_question_id, inspection_language, inspection_strategy, inspection_reference = reference_for_record(inspection_record, reference_by_key, request_by_id)
inspection_response = inspection_record.get("response") or {}
inspection_body = inspection_response.get("body") or {}
inspection_text = response_text(inspection_record)
inspection_answer = parse_model_answer(inspection_text)
inspection_expected = {key: inspection_reference.get(key, "") for key in ("subject", "predicate", "object")}
inspection_slots = {key: answers_match(inspection_answer.get(key), inspection_expected[key], key) for key in inspection_expected}
inspection_request = request_by_id.get(inspection_custom_id, {})
inspection_usage = inspection_body.get("usage") or inspection_response.get("usageMetadata") or {}
inspection_created = inspection_body.get("created_at")
inspection_completed = inspection_body.get("completed_at")

print("Custom ID:", inspection_custom_id)
print("Question ID:", inspection_question_id)
print("Language:", inspection_language)
print("Strategy:", inspection_strategy)
print("Question:\n", inspection_reference.get("question", "<reference not found>"))
print("Story:", inspection_reference.get("story_id", "<reference not found>"))
print("\nCorrect:", all(inspection_slots.values()) if inspection_reference else "unknown (reference not found)")
print("Slot correctness:", inspection_slots)
print("\nExpected:", json.dumps(inspection_expected, ensure_ascii=False))
print("Predicted:", json.dumps(inspection_answer, ensure_ascii=False))
print("Raw model response:", inspection_text or "<empty>")
print("\nResponse metadata:")
print(json.dumps({
    "status_code": inspection_response.get("status_code"), "status": inspection_body.get("status"),
    "response_id": inspection_body.get("id"), "request_id": inspection_response.get("request_id"),
    "model": inspection_body.get("model"), "created_at": inspection_created, "completed_at": inspection_completed,
    "duration_seconds": inspection_completed - inspection_created if isinstance(inspection_created, (int, float)) and isinstance(inspection_completed, (int, float)) else None,
    "usage": inspection_usage, "error": inspection_record.get("error"),
}, ensure_ascii=False, indent=2))

request_body = inspection_request.get("body") or {}

Aantal foutieve voorspellingen: 786 van de 1440


,custom_id,language,strategy,expected,predicted
0,Marten_01_1__nl__zero_shot,nl,zero_shot,"{'subject': 'vrome vrouw', 'predicate': 'livesIn', 'object': 'bale'}","{'subject': 'de zeer vrome vrouw', 'predicate': 'livesIn', 'object': 'unknown'}"
2,Marten_01_1__nl__one_shot,nl,one_shot,"{'subject': 'vrome vrouw', 'predicate': 'livesIn', 'object': 'bale'}","{'subject': 'http://example.org/verhalenbank/Character', 'predicate': 'livesIn', 'object': 'http://example.org/verhalenbank/Location'}"
4,Marten_01_1__nl__few_shot,nl,few_shot,"{'subject': 'vrome vrouw', 'predicate': 'livesIn', 'object': 'bale'}","{'subject': 'Item_49828', 'predicate': 'unknown', 'object': 'unknown'}"
6,Marten_01_1__en__zero_shot,en,zero_shot,"{'subject': 'pious woman', 'predicate': 'livesIn', 'object': 'bale'}","{'subject': 'unknown', 'predicate': 'hasLocation', 'object': 'unknown'}"
8,Marten_01_1__en__one_shot,en,one_shot,"{'subject': 'pious woman', 'predicate': 'livesIn', 'object': 'bale'}","{'subject': 'Item_49828', 'predicate': 'unknown', 'object': 'unknown'}"
...,...,...,...,...,...
1435,Fleur_10_2__fy__zero_shot_ontology,fy,zero_shot_ontology,"{'subject': 'wurdt ferteld', 'predicate': 'hasLocation', 'object': 'herberch 'de Swarte Hûn''}","{'subject': 'unknown', 'predicate': 'unknown', 'object': 'unknown'}"
1436,Fleur_10_2__fy__one_shot,fy,one_shot,"{'subject': 'wurdt ferteld', 'predicate': 'hasLocation', 'object': 'herberch 'de Swarte Hûn''}","{'subject': 'Item_125828', 'predicate': 'hasLocation', 'object': 'herberg 'de Zwarte Hond''}"
1437,Fleur_10_2__fy__one_shot_ontology,fy,one_shot_ontology,"{'subject': 'wurdt ferteld', 'predicate': 'hasLocation', 'object': 'herberch 'de Swarte Hûn''}","{'subject': 'Item_125828', 'predicate': 'hasLocation', 'object': 'herberg_de_Zwarte_Hond'}"
1438,Fleur_10_2__fy__few_shot,fy,few_shot,"{'subject': 'wurdt ferteld', 'predicate': 'hasLocation', 'object': 'herberch 'de Swarte Hûn''}","{'subject': 'Jan de Lapper', 'predicate': 'hasLocation', 'object': 'herberg 'de Zwarte Hond''}"


Custom ID: Marten_01_1__nl__zero_shot
Question ID: Marten_01_1
Language: nl
Strategy: zero_shot
Question:
 Waar komt de zeer vrome vrouw vandaan?
Story: Item_49828

Correct: False
Slot correctness: {'subject': True, 'predicate': True, 'object': False}

Expected: {"subject": "vrome vrouw", "predicate": "livesIn", "object": "bale"}
Predicted: {"subject": "de zeer vrome vrouw", "predicate": "livesIn", "object": "unknown"}
Raw model response: {
  "subject": "de zeer vrome vrouw",
  "predicate": "livesIn",
  "object": "unknown"
}

Response metadata:
{
  "status_code": null,
  "status": null,
  "response_id": null,
  "request_id": null,
  "model": null,
  "created_at": null,
  "completed_at": null,
  "duration_seconds": null,
  "usage": {
    "promptTokenCount": 1794,
    "candidatesTokenCount": 32,
    "totalTokenCount": 1826,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 1794
      }
    ],
    "serviceTier": "SERVICE_TIER_STANDARD"
  },
  "error": 